#WEEK 9 UPDATE

So far, I've built a baseline keyword-matching system for the Menu Analyzer that checks menu items against a hand-built tiered dictionary of dairy-related terms (direct, derivatives, ambiguous, and context clues). I evaluated it against a 30-item hand-labeled test set of real menu items and achieved 90% overall accuracy, with clear failure modes on dishes whose names imply dairy without stating it (e.g., tiramisu, margherita pizza, chicken parmigiana). To strengthen the dictionary's grounding, I integrated the OpenFoodFacts API to pull ingredient text from products tagged with the dairy allergen, mining it for terms my hand-built list had missed. The next step is to extend the system beyond text-only input by accepting photos of physical menus — using OCR to extract menu text from images — and then fine-tuning a BERT model on ingredient-to-allergen pairs so the classifier can recognize hidden dairy derivatives (like "malt-based batter" or "whey protein isolate") without needing them explicitly in a keyword dictionary. This will let me compare three approaches on the same test set: keyword matcher, API-expanded keyword matcher, and fine-tuned BERT — and document which approach handles which failure modes best.

## Part 1: Allergen Dictionary

The dictionary is organized into four tiers so the matcher can reason about **confidence**, not just presence:

- **`direct`** — obvious mentions (`milk`, `cheese`, `butter`). A hit here is enough to flag **UNSAFE**.
- **`derivatives`** — hidden but deterministic (`ghee`, `whey`, `paneer`, `alfredo`). If present, the dish contains the allergen — even though a casual reader might miss it.
- **`ambiguous`** — depends on preparation (`mashed potatoes`, `risotto`, `pesto`). Flag **UNCERTAIN** and surface the reason so the user can ask the restaurant.
- **`context_clues`** — soft phrases that hint at dairy without naming it (`buttery`, `three-cheese`, `drizzled with`). Weight these lower than direct matches.

**Why this structure matters:** the matcher returns `(verdict, reason, tier_hit)`, so later we can measure *where* it fails. Easy wins live in the `direct` tier; the LLM upgrade has to earn its keep on `derivatives` and `ambiguous`.

In [1]:
from typing import Optional

DAIRY = {
    "direct": [
        "milk",
        "cream",
        "heavy cream",
        "half and half",
        "cheese",
        "butter",
        "yogurt",
        "ice cream",
        "sour cream",
        "cream cheese",
        "condensed milk",
        "evaporated milk",
        "buttermilk",
    ],
    "derivatives": [
        "ghee",
        "casein",
        "caseinate",
        "whey",
        "lactose",
        "lactalbumin",
        "lactoglobulin",
        "curd",
        "paneer",
        "ricotta",
        "mascarpone",
        "mozzarella",
        "parmesan",
        "parmigiano",
        "pecorino",
        "gouda",
        "cheddar",
        "feta",
        "brie",
        "camembert",
        "gruyere",
        "halloumi",
        "queso",
        "queso fresco",
        "cotija",
        "crema",
        "crème fraîche",
        "creme fraiche",
        "clotted cream",
        "gelato",
        "custard",
        "béchamel",
        "bechamel",
        "alfredo",
        "carbonara",
        "au gratin",
        "au poivre",
        "quiche",
        "tzatziki",
        "labneh",
        "kefir",
        "ranch",
        "caesar",
    ],
    "ambiguous": [
        "mashed potatoes",
        "scrambled eggs",
        "omelet",
        "pancakes",
        "waffles",
        "french toast",
        "bread",
        "biscuits",
        "croissant",
        "risotto",
        "polenta",
        "pesto",
        "soup",
        "bisque",
        "chowder",
        "sauce",
        "gravy",
        "sautéed",
        "sauteed",
        "pan-fried",
        "glazed",
        "mashed",
        "creamy",
    ],
    "context_clues": [
        "house-made butter",
        "finished with butter",
        "topped with cheese",
        "cheesy",
        "milky",
        "buttery",
        "with a dollop of",
        "drizzled with",
        "cream-based",
        "dairy",
        "three-cheese",
        "four-cheese",
        "cheese blend",
    ],
}

DAIRY_FLAT = {}
for tier, terms in DAIRY.items():
    for term in terms:
        DAIRY_FLAT[term.lower()] = tier

ALLERGENS = {
    "dairy": DAIRY,
}


def get_tier(term: str, restriction: str = "dairy") -> Optional[str]:
    """Return which tier a term belongs to, or None if not in the dictionary."""
    term = term.lower().strip()
    restriction_dict = ALLERGENS.get(restriction, {})
    for tier, terms in restriction_dict.items():
        if term in [t.lower() for t in terms]:
            return tier
    return None


print(f"Dairy direct terms: {len(DAIRY['direct'])}")
print(f"Dairy derivatives: {len(DAIRY['derivatives'])}")
print(f"Dairy ambiguous: {len(DAIRY['ambiguous'])}")
print(f"Dairy context clues: {len(DAIRY['context_clues'])}")
print(f"Total dairy terms: {len(DAIRY_FLAT)}")
print()
print(f"'ghee' is in tier: {get_tier('ghee')}")
print(f"'mashed potatoes' is in tier: {get_tier('mashed potatoes')}")
print(f"'grilled chicken' is in tier: {get_tier('grilled chicken')}")

Dairy direct terms: 13
Dairy derivatives: 43
Dairy ambiguous: 23
Dairy context clues: 13
Total dairy terms: 92

'ghee' is in tier: derivatives
'mashed potatoes' is in tier: ambiguous
'grilled chicken' is in tier: None


## Part 2: Test Set

We hand-label ~30 real menu items (taken from actual restaurant menus) with ground-truth `SAFE` / `UNSAFE` / `UNCERTAIN` labels. Hand-labeling matters because:

- **Realism** — synthetic items leak the dictionary's vocabulary back into the test set, inflating accuracy. Real menu prose doesn't.
- **Difficulty tags** (`easy` / `medium` / `hard`) let us report accuracy *per tier of difficulty*, not a single number that hides failure modes.
- **Category tags** (e.g. `italian`, `indian`, `breakfast`) let us spot systematic blind spots by cuisine — e.g. Indian dishes where dairy hides as `paneer` or `ghee`.

> **TODO (stub):** The actual 30 labeled items haven't been authored yet. The cell below defines an empty `TEST_ITEMS` list and a `summary()` function so downstream cells still run.

In [2]:
# 30 hand-labeled menu items. Ground-truth labels reflect what the
# typical version of the dish contains, not just what the description names.

TEST_ITEMS = [
    # --- easy UNSAFE (direct keyword in description) ---
    {"name": "Cheeseburger", "description": "Grilled beef patty with cheddar on a brioche bun", "label": "UNSAFE", "difficulty": "easy", "category": "american"},
    {"name": "Fettuccine Alfredo", "description": "Fettuccine in rich alfredo sauce with parmesan", "label": "UNSAFE", "difficulty": "easy", "category": "italian"},
    {"name": "Spaghetti Carbonara", "description": "Spaghetti with eggs, pancetta, and pecorino", "label": "UNSAFE", "difficulty": "easy", "category": "italian"},
    {"name": "Caesar Salad", "description": "Romaine with caesar dressing, croutons, anchovies", "label": "UNSAFE", "difficulty": "easy", "category": "american"},
    {"name": "Greek Gyro", "description": "Seasoned lamb with tzatziki in pita", "label": "UNSAFE", "difficulty": "easy", "category": "greek"},
    {"name": "French Onion Soup", "description": "Caramelized onions in broth topped with gruyere", "label": "UNSAFE", "difficulty": "easy", "category": "french"},
    {"name": "Quiche Lorraine", "description": "Savory tart with bacon, eggs, and gruyere", "label": "UNSAFE", "difficulty": "easy", "category": "french"},
    {"name": "Crème Brûlée", "description": "Classic French custard with caramelized sugar top", "label": "UNSAFE", "difficulty": "easy", "category": "french"},
    {"name": "Butter Chicken", "description": "Tandoori chicken in tomato and cream sauce", "label": "UNSAFE", "difficulty": "easy", "category": "indian"},

    # --- medium UNSAFE (derivative tier) ---
    {"name": "Palak Paneer", "description": "Spinach curry with cubes of fresh paneer", "label": "UNSAFE", "difficulty": "medium", "category": "indian"},
    {"name": "Chicken Biryani", "description": "Basmati rice layered with spiced chicken and ghee", "label": "UNSAFE", "difficulty": "medium", "category": "indian"},
    {"name": "Tiramisu", "description": "Espresso-soaked ladyfingers layered with mascarpone", "label": "UNSAFE", "difficulty": "medium", "category": "italian"},
    {"name": "Dal Makhani", "description": "Creamy black lentils simmered overnight", "label": "UNSAFE", "difficulty": "medium", "category": "indian"},

    # --- hard UNSAFE (implied by dish name; keyword matcher expected to FAIL) ---
    {"name": "Margherita Pizza", "description": "Classic Neapolitan margherita with fresh basil", "label": "UNSAFE", "difficulty": "hard", "category": "italian"},
    {"name": "Chicken Parmigiana", "description": "Breaded chicken cutlet with marinara, baked golden", "label": "UNSAFE", "difficulty": "hard", "category": "italian"},
    {"name": "Spaghetti Bolognese", "description": "Spaghetti with slow-cooked beef and tomato ragu", "label": "UNSAFE", "difficulty": "hard", "category": "italian"},
    {"name": "Chocolate Brownie", "description": "Fudgy chocolate brownie with walnuts", "label": "UNSAFE", "difficulty": "hard", "category": "dessert"},

    # --- UNCERTAIN (ambiguous preparation) ---
    {"name": "Mashed Potatoes", "description": "Yukon gold mashed potatoes with herbs", "label": "UNCERTAIN", "difficulty": "medium", "category": "american"},
    {"name": "Pesto Pasta", "description": "Linguine tossed with fresh basil pesto", "label": "UNCERTAIN", "difficulty": "medium", "category": "italian"},
    {"name": "Risotto ai Funghi", "description": "Arborio rice with wild mushrooms and white wine", "label": "UNCERTAIN", "difficulty": "medium", "category": "italian"},

    # --- easy SAFE ---
    {"name": "Grilled Salmon", "description": "Wild salmon with lemon and olive oil", "label": "SAFE", "difficulty": "easy", "category": "american"},
    {"name": "Garden Salad", "description": "Mixed greens, cucumber, tomato, balsamic vinaigrette", "label": "SAFE", "difficulty": "easy", "category": "american"},
    {"name": "Falafel Wrap", "description": "Chickpea patties with tahini in pita", "label": "SAFE", "difficulty": "easy", "category": "mediterranean"},
    {"name": "Hummus Plate", "description": "Chickpea dip with olive oil and pita", "label": "SAFE", "difficulty": "easy", "category": "mediterranean"},
    {"name": "Pad Thai", "description": "Rice noodles with peanuts, egg, shrimp, tamarind", "label": "SAFE", "difficulty": "easy", "category": "thai"},
    {"name": "Vegetable Stir-Fry", "description": "Mixed vegetables wok-fried with soy and garlic", "label": "SAFE", "difficulty": "easy", "category": "asian"},
    {"name": "Avocado Toast", "description": "Sourdough with smashed avocado and chili flakes", "label": "SAFE", "difficulty": "easy", "category": "breakfast"},

    # --- medium SAFE (matcher may over-flag as UNCERTAIN from the ambiguous tier: 'soup', 'gravy') ---
    {"name": "Tom Yum Soup", "description": "Spicy Thai broth with shrimp and lemongrass", "label": "SAFE", "difficulty": "medium", "category": "thai"},
    {"name": "Pho", "description": "Vietnamese beef noodle soup with fresh herbs", "label": "SAFE", "difficulty": "medium", "category": "vietnamese"},
    {"name": "Miso Soup", "description": "Japanese broth with tofu and seaweed", "label": "SAFE", "difficulty": "medium", "category": "japanese"},
]


def summary():
    """Print a breakdown of the test set by label / difficulty / category."""
    if not TEST_ITEMS:
        print("TEST_ITEMS is empty (stub). Add hand-labeled items to populate.")
        return
    from collections import Counter
    print(f"Total items: {len(TEST_ITEMS)}")
    print(f"By label:      {dict(Counter(i['label'] for i in TEST_ITEMS))}")
    print(f"By difficulty: {dict(Counter(i['difficulty'] for i in TEST_ITEMS))}")
    print(f"By category:   {dict(Counter(i['category'] for i in TEST_ITEMS))}")


summary()

Total items: 30
By label:      {'UNSAFE': 17, 'UNCERTAIN': 3, 'SAFE': 10}
By difficulty: {'easy': 16, 'medium': 10, 'hard': 4}
By category:   {'american': 5, 'italian': 8, 'greek': 1, 'french': 3, 'indian': 4, 'dessert': 1, 'mediterranean': 2, 'thai': 2, 'asian': 1, 'breakfast': 1, 'vietnamese': 1, 'japanese': 1}


## Part 3: The Keyword Matcher

This is the **baseline**. A dumb keyword matcher that scans menu text for terms in our tiered dictionary and returns a verdict plus the reason behind it.

The verdict logic:
- `direct` or `derivatives` hit &rarr; **UNSAFE**
- only `ambiguous` or `context_clues` hits &rarr; **UNCERTAIN**
- no hits &rarr; **SAFE**

Later we'll compare this against an LLM-backed matcher on the same test set. The gap between them is the whole point of the project.

> **TODO (stub):** The cell below is a minimal skeleton so downstream demo and evaluation cells still run. A real implementation should use word-boundary matching (not bare `in`) to avoid false positives like `cream` matching `creamy_vegan_sauce`.

In [3]:
# STUB: simple substring-based matcher. Replace with word-boundary matching
# (e.g. regex with \b) and a smarter reason-builder.

def check_item(text: str, restriction: str = "dairy"):
    """Return {verdict, reason, tier_hit, matched_terms} for a menu item."""
    text_lower = text.lower()
    restriction_dict = ALLERGENS.get(restriction, {})

    hits = {"direct": [], "derivatives": [], "ambiguous": [], "context_clues": []}
    for tier, terms in restriction_dict.items():
        for term in terms:
            if term.lower() in text_lower:
                hits[tier].append(term)

    if hits["direct"] or hits["derivatives"]:
        verdict = "UNSAFE"
        tier_hit = "direct" if hits["direct"] else "derivatives"
    elif hits["ambiguous"] or hits["context_clues"]:
        verdict = "UNCERTAIN"
        tier_hit = "ambiguous" if hits["ambiguous"] else "context_clues"
    else:
        verdict = "SAFE"
        tier_hit = None

    matched = [t for tier_hits in hits.values() for t in tier_hits]
    return {
        "verdict": verdict,
        "reason": build_reason(verdict, hits),
        "tier_hit": tier_hit,
        "matched_terms": matched,
    }


def build_reason(verdict, hits):
    """Build a human-readable reason string from the hit dict."""
    if verdict == "SAFE":
        return "No dairy terms detected."
    parts = []
    for tier in ("direct", "derivatives", "ambiguous", "context_clues"):
        if hits[tier]:
            parts.append(f"{tier}: {', '.join(hits[tier])}")
    return "; ".join(parts)


def format_result(item_name, result):
    """Pretty-print a single check_item result."""
    return (
        f"[{result['verdict']:<9}] {item_name}\n"
        f"    reason: {result['reason']}"
    )

## Part 4: Demo

A quick visual check on five items that span the difficulty spectrum: an obvious UNSAFE, a hidden-derivative UNSAFE, an ambiguous case, a SAFE, and a known blind spot.

In [4]:
demo_items = [
    ("Cheeseburger", "Grilled beef patty with cheddar on a brioche bun"),
    ("Palak Paneer", "Spinach curry with cubes of fresh paneer"),
    ("Mashed Potatoes", "Yukon gold mashed potatoes with herbs"),
    ("Grilled Salmon", "Wild salmon with lemon and olive oil"),
    ("Tiramisu", "Espresso-soaked ladyfingers layered with mascarpone"),
]

for name, desc in demo_items:
    result = check_item(desc)
    print(format_result(name, result))
    print()

[UNSAFE   ] Cheeseburger
    reason: derivatives: cheddar

[UNSAFE   ] Palak Paneer
    reason: derivatives: paneer

[UNCERTAIN] Mashed Potatoes
    reason: ambiguous: mashed potatoes, mashed

[SAFE     ] Grilled Salmon
    reason: No dairy terms detected.

[UNSAFE   ] Tiramisu
    reason: derivatives: mascarpone



## Part 5: Evaluation

How we measure accuracy. Every future version of the system (LLM layer, hybrid, etc.) gets scored against the **same** `TEST_ITEMS` set so we can compare apples to apples.

We report:
- overall accuracy,
- accuracy broken down by difficulty,
- accuracy broken down by ground-truth label (so we can see if the matcher is biased toward SAFE or UNSAFE),
- a confusion matrix.

> **TODO (stub):** The cell below handles an empty `TEST_ITEMS` gracefully. Once the test set is populated, the report becomes meaningful.

In [5]:
def evaluate(items=None):
    """Run check_item on each test item and return per-item predictions + labels."""
    items = items if items is not None else TEST_ITEMS
    results = []
    for item in items:
        pred = check_item(item["description"])
        results.append({
            "name": item["name"],
            "expected": item["label"],
            "predicted": pred["verdict"],
            "correct": pred["verdict"] == item["label"],
            "difficulty": item.get("difficulty", "?"),
            "category": item.get("category", "?"),
            "reason": pred["reason"],
        })
    return results


def print_report(results):
    """Print overall / by-difficulty / by-label accuracy plus a confusion matrix."""
    if not results:
        print("No test items to evaluate (stub). Populate TEST_ITEMS to get a real report.")
        return

    from collections import Counter, defaultdict

    total = len(results)
    correct = sum(1 for r in results if r["correct"])
    print(f"Overall accuracy: {correct}/{total} = {correct/total:.1%}")
    print()

    by_diff = defaultdict(list)
    for r in results:
        by_diff[r["difficulty"]].append(r["correct"])
    print("By difficulty:")
    for diff, outcomes in sorted(by_diff.items()):
        n, c = len(outcomes), sum(outcomes)
        print(f"  {diff:<8} {c}/{n} = {c/n:.1%}")
    print()

    by_label = defaultdict(list)
    for r in results:
        by_label[r["expected"]].append(r["correct"])
    print("By ground-truth label:")
    for label, outcomes in sorted(by_label.items()):
        n, c = len(outcomes), sum(outcomes)
        print(f"  {label:<10} {c}/{n} = {c/n:.1%}")
    print()

    confusion = Counter((r["expected"], r["predicted"]) for r in results)
    labels = sorted({r["expected"] for r in results} | {r["predicted"] for r in results})
    print("Confusion matrix (rows = expected, cols = predicted):")
    header = "           " + "  ".join(f"{l:>9}" for l in labels)
    print(header)
    for exp in labels:
        row = f"  {exp:<9}" + "  ".join(f"{confusion.get((exp, p), 0):>9}" for p in labels)
        print(row)
    print()

    wrong = [r for r in results if not r["correct"]]
    if wrong:
        print(f"Misclassified items ({len(wrong)}):")
        for r in wrong:
            print(f"  - {r['name']}: expected {r['expected']}, got {r['predicted']}")
            print(f"      reason: {r['reason']}")

In [6]:
results = evaluate()
print_report(results)

Overall accuracy: 23/30 = 76.7%

By difficulty:
  easy     15/16 = 93.8%
  hard     0/4 = 0.0%
  medium   8/10 = 80.0%

By ground-truth label:
  SAFE       8/10 = 80.0%
  UNCERTAIN  2/3 = 66.7%
  UNSAFE     13/17 = 76.5%

Confusion matrix (rows = expected, cols = predicted):
                SAFE  UNCERTAIN     UNSAFE
  SAFE             8          2          0
  UNCERTAIN        1          2          0
  UNSAFE           3          1         13

Misclassified items (7):
  - Margherita Pizza: expected UNSAFE, got SAFE
      reason: No dairy terms detected.
  - Chicken Parmigiana: expected UNSAFE, got UNCERTAIN
      reason: ambiguous: bread
  - Spaghetti Bolognese: expected UNSAFE, got SAFE
      reason: No dairy terms detected.
  - Chocolate Brownie: expected UNSAFE, got SAFE
      reason: No dairy terms detected.
  - Risotto ai Funghi: expected UNCERTAIN, got SAFE
      reason: No dairy terms detected.
  - Avocado Toast: expected SAFE, got UNCERTAIN
      reason: ambiguous: mashed
  - 

## Findings So Far

*Placeholder — fill in once `TEST_ITEMS` is populated and the evaluation has been run.*

Specific failure modes to look for:
- **Tiramisu** — `mascarpone` is in the `derivatives` tier, so this *should* be caught. If it isn't, the bug is in the matcher, not the dictionary.
- **Margherita pizza** — the word "margherita" doesn't contain any dairy token, but the dish is defined by mozzarella. Keyword matching can only catch this if the description mentions `mozzarella` / `cheese` explicitly.
- **Chicken parmigiana** — `parmigiano` is a derivative, but `parmigiana` (the dish) may not be in the dictionary. A spelling/morphology mismatch.
- **Indian dishes with ghee** — `ghee` is in the dictionary, but menus often don't name it ("seasoned rice," "tempered with spices").

**What this suggests for next steps:**
- A keyword matcher can only detect what's named. The interesting failures are dishes where the dairy is *implied by the name alone* (margherita, carbonara, parmigiana).
- An LLM layer should earn its keep here — it has world knowledge about what dishes *typically* contain, not just what's in the description string.
- The right metric isn't overall accuracy — it's accuracy on the `hard` and `derivatives`-dominated subset, where the LLM has room to win.

## Part 6: Expanding the Dictionary with OpenFoodFacts

OpenFoodFacts is a free, crowdsourced database of 2M+ food products with ingredient
and allergen labels. We query it for products tagged with the dairy (en:milk) allergen,
pull their ingredient text, and mine for dairy-related terms our hand-built dictionary
might have missed. This gives our dictionary a defensible, citable data source instead
of relying on pure intuition.

Data license: Open Database License (ODbL). Attribution required.

In [7]:
import requests
from typing import List, Optional

USER_AGENT = "MenuAnalyzer/0.1 - CS35 Final Project - educational use"
OFF_SEARCH_URL = "https://world.openfoodfacts.org/api/v2/search"


def fetch_dairy_products(page_count: int = 3, page_size: int = 100) -> List[str]:
    """Fetch ingredient_text strings from OpenFoodFacts products tagged en:milk.

    Returns whatever was collected so far on timeout or HTTP error — never crashes.
    """
    headers = {"User-Agent": USER_AGENT}
    collected: List[str] = []
    total_products = 0

    for page in range(1, page_count + 1):
        params = {
            "allergens_tags": "en:milk",
            "fields": "product_name,ingredients_text,allergens_tags",
            "page_size": page_size,
            "page": page,
        }
        try:
            resp = requests.get(OFF_SEARCH_URL, params=params, headers=headers, timeout=10)
            resp.raise_for_status()
            data = resp.json()
        except requests.exceptions.Timeout:
            print(f"  page {page}: TIMEOUT after 10s — returning {len(collected)} ingredient texts collected so far")
            break
        except requests.exceptions.RequestException as err:
            print(f"  page {page}: ERROR {err} — returning {len(collected)} ingredient texts collected so far")
            break

        products = data.get("products", [])
        total_products += len(products)
        page_texts = [
            p["ingredients_text"]
            for p in products
            if isinstance(p.get("ingredients_text"), str) and p["ingredients_text"].strip()
        ]
        collected.extend(page_texts)
        print(f"  page {page}: fetched {len(products)} products, {len(page_texts)} with ingredient text")

    print(f"\nTotal products seen: {total_products}. Ingredient texts collected: {len(collected)}.")
    return collected


/Users/aariachandwani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [8]:
ingredient_texts = fetch_dairy_products(page_count=3)

print(f"\nTotal products fetched (with ingredients): {len(ingredient_texts)}")
print("\nSample — first 2 ingredient texts:")
for i, text in enumerate(ingredient_texts[:2], 1):
    print(f"\n--- Sample {i} ---")
    print(text)


  page 1: fetched 100 products, 100 with ingredient text


  page 2: ERROR 503 Server Error: Service Temporarily Unavailable for url: https://world.openfoodfacts.org/api/v2/search?allergens_tags=en%3Amilk&fields=product_name%2Cingredients_text%2Callergens_tags&page_size=100&page=2 — returning 100 ingredient texts collected so far

Total products seen: 100. Ingredient texts collected: 100.

Total products fetched (with ingredients): 100

Sample — first 2 ingredient texts:

--- Sample 1 ---
Lait de Vache pasteurisé, Crème fraiche pasteurisée, Ferments lactiques, Présure, Sel.

--- Sample 2 ---
milk cream, cream, sugar, banana, bacteria


In [9]:
import re
from collections import Counter

STOPWORDS = {
    "the", "and", "with", "from", "water", "salt", "sugar", "oil", "flour",
    "egg", "eggs", "wheat", "soy", "corn", "rice", "starch", "acid", "natural",
    "flavor", "flavors", "contains", "may", "preservative", "color", "extract",
    "powder", "organic", "processed",
}


def extract_candidate_terms(texts: List[str], top_n: int = 50):
    """Return up-to-top_n most frequent tokens that are NOT already in DAIRY_FLAT.

    Tokens under 3 chars, pure digits, and common stopwords are dropped.
    """
    combined = " ".join(texts).lower()
    tokens = re.split(r"[,.;:()\[\]\-\s]+", combined)

    filtered = []
    for tok in tokens:
        tok = tok.strip()
        if len(tok) < 3:
            continue
        if tok.isdigit():
            continue
        if tok in STOPWORDS:
            continue
        filtered.append(tok)

    counts = Counter(filtered)
    candidates = [(term, freq) for term, freq in counts.most_common() if term not in DAIRY_FLAT]
    return candidates[:top_n]


In [10]:
candidates = extract_candidate_terms(ingredient_texts, top_n=50)

header = f"{'Candidate Term':<30} {'Frequency':>10}"
print(header)
print("-" * len(header))
for term, freq in candidates:
    print(f"{term:<30} {freq:>10}")


Candidate Term                  Frequency
-----------------------------------------
lait                                   96
sucre                                  74
farine                                 70
poudre                                 69
blé                                    54
sel                                    49
cacao                                  49
écrémé                                 45
huile                                  43
soja                                   36
beurre                                 32
arôme                                  29
carbonates                             25
émulsifiant                            24
lécithines                             24
crème                                  23
colza                                  23
naturel                                23
pâte                                   22
palme                                  21
ferments                               20
frais                             

## Manual Review Required

The API surfaces candidate terms, but not all are dairy-related. For example, 'sugar'
might appear frequently in dairy-containing products but isn't itself a dairy marker.
Review the list above and paste confirmed dairy-related terms into the NEW_DAIRY_TERMS
list in the next cell.

In [11]:
# After reviewing the candidates above, paste confirmed dairy-related terms here.
# Example: NEW_DAIRY_TERMS = ["skimmed milk powder", "dried whey", "milk protein"]
NEW_DAIRY_TERMS = []


def add_terms_to_dictionary(new_terms, tier="derivatives"):
    """Extend the existing DAIRY dictionary with reviewed terms."""
    added = []
    for term in new_terms:
        term_lower = term.lower().strip()
        if term_lower and term_lower not in DAIRY_FLAT:
            DAIRY[tier].append(term_lower)
            DAIRY_FLAT[term_lower] = tier
            added.append(term_lower)
    print(f"Added {len(added)} new terms to tier '{tier}': {added}")
    return added


if NEW_DAIRY_TERMS:
    add_terms_to_dictionary(NEW_DAIRY_TERMS)
else:
    print("NEW_DAIRY_TERMS is empty — review candidates and add confirmed terms above.")


NEW_DAIRY_TERMS is empty — review candidates and add confirmed terms above.


In [12]:
results = evaluate()
print_report(results)


Overall accuracy: 23/30 = 76.7%

By difficulty:
  easy     15/16 = 93.8%
  hard     0/4 = 0.0%
  medium   8/10 = 80.0%

By ground-truth label:
  SAFE       8/10 = 80.0%
  UNCERTAIN  2/3 = 66.7%
  UNSAFE     13/17 = 76.5%

Confusion matrix (rows = expected, cols = predicted):
                SAFE  UNCERTAIN     UNSAFE
  SAFE             8          2          0
  UNCERTAIN        1          2          0
  UNSAFE           3          1         13

Misclassified items (7):
  - Margherita Pizza: expected UNSAFE, got SAFE
      reason: No dairy terms detected.
  - Chicken Parmigiana: expected UNSAFE, got UNCERTAIN
      reason: ambiguous: bread
  - Spaghetti Bolognese: expected UNSAFE, got SAFE
      reason: No dairy terms detected.
  - Chocolate Brownie: expected UNSAFE, got SAFE
      reason: No dairy terms detected.
  - Risotto ai Funghi: expected UNCERTAIN, got SAFE
      reason: No dairy terms detected.
  - Avocado Toast: expected SAFE, got UNCERTAIN
      reason: ambiguous: mashed
  - 

## Source Attribution

Dictionary expansion candidates sourced from the OpenFoodFacts database
(https://world.openfoodfacts.org), queried via API v2 on 2026-04-20.
Data licensed under the Open Database License (ODbL) v1.0.
Database made available by Open Food Facts contributors.

## Part 7: Multi-restriction Matcher

The `check_item` matcher in Part 3 is dairy-only. Here we generalize to **N restrictions at once** plus diet bundles (`vegan`, `vegetarian`, `pescatarian`).

Design choices:
- **Same 4-tier structure** for every restriction (`direct` / `derivatives` / `ambiguous` / `context_clues`) so logic and reasoning stay uniform.
- **Empty placeholder dictionaries** for restrictions we haven't built out yet (gluten, nuts, eggs, soy, sesame, animal_derived). They return `safe` for everything until populated — that's a feature, not a bug, since it lets downstream code work today.
- **Diet bundles expand to atomic restrictions** before matching — so `vegetarian` becomes `[meat, fish, shellfish]`, and the per-restriction dict shows the *real* trigger, not the bundle name.
- **`needs_llm_review` only fires on `uncertain`**. Items already known unsafe are settled — no point spending an LLM call on them.

In [13]:
# --- 1. Empty placeholder dictionaries (same 4-tier shape as DAIRY) ---
def _empty_tiers():
    return {"direct": [], "derivatives": [], "ambiguous": [], "context_clues": []}

GLUTEN = _empty_tiers()
NUTS = _empty_tiers()
EGGS = _empty_tiers()
SOY = _empty_tiers()
SESAME = _empty_tiers()
ANIMAL_DERIVED = _empty_tiers()

# --- 2. Populated direct tier for meat / fish / shellfish (just basics, not exhaustive) ---
MEAT = _empty_tiers()
MEAT["direct"] = [
    "chicken", "beef", "pork", "lamb", "turkey", "duck", "veal",
    "prosciutto", "bacon", "sausage", "ham",
]

FISH = _empty_tiers()
FISH["direct"] = [
    "salmon", "tuna", "branzino", "cod", "halibut", "sea bass", "anchovy", "sardine",
]

SHELLFISH = _empty_tiers()
SHELLFISH["direct"] = [
    "shrimp", "prawn", "lobster", "crab", "scallop", "oyster", "mussel", "clam",
    "calamari", "squid",
]

# Register without disturbing DAIRY (already in ALLERGENS from Part 1)
ALLERGENS["gluten"] = GLUTEN
ALLERGENS["nuts"] = NUTS
ALLERGENS["eggs"] = EGGS
ALLERGENS["soy"] = SOY
ALLERGENS["sesame"] = SESAME
ALLERGENS["meat"] = MEAT
ALLERGENS["fish"] = FISH
ALLERGENS["shellfish"] = SHELLFISH
ALLERGENS["animal_derived"] = ANIMAL_DERIVED


# --- 3. Diet bundles ---
DIET_BUNDLES = {
    "pescatarian": ["meat"],
    "vegetarian": ["meat", "fish", "shellfish"],
    "vegan": ["meat", "fish", "shellfish", "dairy", "eggs", "animal_derived"],
}


# --- 4. Functions ---
from typing import List, Dict, Any


def expand_restrictions(selected: List[str]) -> List[str]:
    """Expand diet bundles into atomic restrictions; deduplicate; preserve order."""
    expanded: List[str] = []
    seen = set()
    for r in selected:
        targets = DIET_BUNDLES.get(r, [r])
        for t in targets:
            if t not in seen:
                expanded.append(t)
                seen.add(t)
    return expanded


def _check_single(text_lower: str, restriction: str) -> Dict[str, Any]:
    """Run one restriction's tiered scan against pre-lowercased text."""
    restriction_dict = ALLERGENS.get(restriction, {})
    hits = {"direct": [], "derivatives": [], "ambiguous": [], "context_clues": []}
    for tier, terms in restriction_dict.items():
        for term in terms:
            if term.lower() in text_lower:
                hits[tier].append(term)

    if hits["direct"] or hits["derivatives"]:
        verdict = "unsafe"
        tier_hit = "direct" if hits["direct"] else "derivatives"
    elif hits["ambiguous"] or hits["context_clues"]:
        verdict = "uncertain"
        tier_hit = "ambiguous" if hits["ambiguous"] else "context_clues"
    else:
        verdict = "safe"
        tier_hit = None

    matched = [t for tier_hits in hits.values() for t in tier_hits]

    if verdict == "safe":
        reason = f"No {restriction} terms detected."
    else:
        parts = [f"{t}: {', '.join(hits[t])}" for t in ("direct", "derivatives", "ambiguous", "context_clues") if hits[t]]
        reason = "; ".join(parts)

    return {
        "verdict": verdict,
        "tier": tier_hit,
        "matched_terms": matched,
        "reason": reason,
    }


def match_item(item: Dict[str, Any], restrictions: List[str]) -> Dict[str, Any]:
    """Check an item against multiple restrictions; return per-restriction + combined verdict."""
    expanded = expand_restrictions(restrictions)
    text_lower = (item.get("name", "") + " " + item.get("description", "")).lower()

    per_restriction: Dict[str, Dict[str, Any]] = {}
    for r in expanded:
        per_restriction[r] = _check_single(text_lower, r)

    verdicts = {pr["verdict"] for pr in per_restriction.values()}
    if "unsafe" in verdicts:
        combined = "unsafe"
    elif "uncertain" in verdicts:
        combined = "uncertain"
    else:
        combined = "safe"

    return {
        "name": item.get("name", ""),
        "description": item.get("description", ""),
        "per_restriction": per_restriction,
        "combined_verdict": combined,
        "needs_llm_review": combined == "uncertain",
    }


def match_menu(menu: Dict[str, Any], restrictions: List[str]) -> Dict[str, Any]:
    """Apply match_item to every item in every section of one menu JSON. Preserve structure."""
    return {
        "restaurant_name": menu.get("restaurant_name"),
        "cuisine": menu.get("cuisine"),
        "location": menu.get("location"),
        "sections": [
            {
                "name": section["name"],
                "items": [match_item(item, restrictions) for item in section["items"]],
            }
            for section in menu.get("sections", [])
        ],
    }


### Verification

Five test cases that cover:
- (a) atomic + bundle: dairy + vegetarian on a meat dish
- (b) two unsafes at once: dairy + meat on chicken parmigiana
- (c) `uncertain` triggers `needs_llm_review`
- (d) bundle-expanded fish trigger
- (e) all-safe across an unfilled dict (gluten) + a populated bundle (vegan)

In [14]:
test_cases = [
    (
        {"name": "Grilled chicken with herbs", "description": "lemon and oregano"},
        ["dairy", "vegetarian"],
        "expect combined='unsafe' — chicken triggers meat (vegetarian bundle)",
    ),
    (
        {"name": "Chicken parmigiana", "description": "breaded chicken with parmesan and mozzarella"},
        ["dairy", "vegetarian"],
        "expect 'unsafe' on BOTH dairy AND meat",
    ),
    (
        {"name": "Risotto al funghi", "description": "wild mushroom risotto"},
        ["dairy"],
        "expect combined='uncertain', needs_llm_review=True (risotto is ambiguous tier)",
    ),
    (
        {"name": "Grilled branzino", "description": "with olive oil and lemon"},
        ["dairy", "vegetarian"],
        "expect 'unsafe' on fish (vegetarian bundle), safe on dairy",
    ),
    (
        {"name": "Mixed green salad", "description": "olive oil vinaigrette"},
        ["dairy", "gluten", "vegan"],
        "expect 'safe' on all (gluten/eggs/animal_derived dicts are empty → safe)",
    ),
]

for i, (item, restrictions, hint) in enumerate(test_cases):
    label = chr(ord("a") + i)
    result = match_item(item, restrictions)
    print(f"=== ({label}) {item['name']} ===")
    print(f"    description     : {item['description']!r}")
    print(f"    restrictions    : {restrictions} → expanded: {expand_restrictions(restrictions)}")
    print(f"    expectation     : {hint}")
    print(f"    combined_verdict: {result['combined_verdict']}")
    print(f"    needs_llm_review: {result['needs_llm_review']}")
    print(f"    per_restriction :")
    for r, pr in result["per_restriction"].items():
        matched = f" matched={pr['matched_terms']}" if pr["matched_terms"] else ""
        print(f"      {r:<16} {pr['verdict']:<10} (tier={pr['tier'] or '—'}){matched}")
    print()


=== (a) Grilled chicken with herbs ===
    description     : 'lemon and oregano'
    restrictions    : ['dairy', 'vegetarian'] → expanded: ['dairy', 'meat', 'fish', 'shellfish']
    expectation     : expect combined='unsafe' — chicken triggers meat (vegetarian bundle)
    combined_verdict: unsafe
    needs_llm_review: False
    per_restriction :
      dairy            safe       (tier=—)
      meat             unsafe     (tier=direct) matched=['chicken']
      fish             safe       (tier=—)
      shellfish        safe       (tier=—)

=== (b) Chicken parmigiana ===
    description     : 'breaded chicken with parmesan and mozzarella'
    restrictions    : ['dairy', 'vegetarian'] → expanded: ['dairy', 'meat', 'fish', 'shellfish']
    expectation     : expect 'unsafe' on BOTH dairy AND meat
    combined_verdict: unsafe
    needs_llm_review: False
    per_restriction :
      dairy            unsafe     (tier=derivatives) matched=['mozzarella', 'parmesan', 'bread']
      meat          